# Utils

> Various utility functions and helpers used across the project.

In [ ]:
#| default_exp utils

In [ ]:
#| hide
from nbdev.showdoc import *

In [10]:
#| export
import numpy as np
import pandas as pd
import torch
import gdown

In [ ]:
#| export
def download_file(url: str, output: str= ".") -> None:
    """Downloads a file from the given URL and saves it to the specified output path.
    Args:
        url (str): The URL to download the file from.
        output (str): The path where the downloaded file should be saved.
    """
    gdown.download(url, output)

### CSI related

In [ ]:
#| export
H_full_url = "https://drive.google.com/file/d/126gvv5GBgzgG21y19fza5vHBtFGxRWMq/view?usp=sharing"
valid_2d_url = "https://drive.google.com/file/d/1Dzj9joHNG434-ZQy3lIEBlscFvw_jH0w/view?usp=sharing"
csi_data_url = "https://drive.google.com/file/d/1bAQEiAvFU-oNeb_UuFO8nOD0NKaL08le/view?usp=sharing"


In [ ]:
#| export
def get_h_full(H_full_url: str = H_full_url) -> np.ndarray:
    download_file(H_full_url, "H_full.npy")
    return np.load("H_full.npy")

def get_valid_2d(valid_2d_url: str = valid_2d_url) -> list:
    download_file(valid_2d_url, "valid_2d.npy")
    return [tuple(p) for p in np.load("valid_2d.npy")]

def get_csi_data(csi_data_url: str = csi_data_url) -> pd.DataFrame:
    download_file(csi_data_url, "csi_data.csv")
    return pd.read_csv("csi_data.csv")


In [ ]:
#| hide
H_full   = get_h_full()
valid_2d = get_valid_2d()
csi_data = get_csi_data()
N = len(valid_2d)

grid_to_idx  = {(gx, gy): i for i, (gx, gy) in enumerate(valid_2d)}
idx_to_grid  = {i: (gx, gy) for i, (gx, gy) in enumerate(valid_2d)}

Downloading...
From: https://drive.google.com/uc?id=126gvv5GBgzgG21y19fza5vHBtFGxRWMq
To: /home/ahmed/Ahmed-home/1- Projects/Research/Journal 2/code/c3jepa-wm/nbs/H_full.npy
100%|██████████| 275k/275k [00:00<00:00, 6.45MB/s]
Downloading...
From: https://drive.google.com/uc?id=1Dzj9joHNG434-ZQy3lIEBlscFvw_jH0w
To: /home/ahmed/Ahmed-home/1- Projects/Research/Journal 2/code/c3jepa-wm/nbs/valid_2d.npy
100%|██████████| 2.22k/2.22k [00:00<00:00, 3.45MB/s]
Downloading...
From: https://drive.google.com/uc?id=1bAQEiAvFU-oNeb_UuFO8nOD0NKaL08le
To: /home/ahmed/Ahmed-home/1- Projects/Research/Journal 2/code/c3jepa-wm/nbs/csi_data.csv
100%|██████████| 1.72M/1.72M [00:00<00:00, 15.3MB/s]


In [17]:
#| hide
csi_data.head()

,tx_grid,rx_grid,csi
0,"(np.int64(1), np.int64(2))","(np.int64(1), np.int64(3))",(0.9994903206825256+0.03192238509654999j)
1,"(np.int64(1), np.int64(2))","(np.int64(1), np.int64(5))",(0.2527361214160919-0.9675353169441223j)
2,"(np.int64(1), np.int64(2))","(np.int64(1), np.int64(8))",(-0.8033327460289001+0.5955304503440857j)
3,"(np.int64(1), np.int64(2))","(np.int64(1), np.int64(9))",(-0.5029078722000122+0.8643400073051453j)
4,"(np.int64(1), np.int64(2))","(np.int64(1), np.int64(10))",(-0.4481883943080902+0.8939390778541565j)


In [ ]:
#| export
def np_get_csi(H_full: np.ndarray, grid_to_idx: dict, tx_grid: tuple, rx_grid: tuple) -> np.ndarray:
    """
    Get CFR for a D2D link between two grid positions.
    tx_grid, rx_grid: (gx, gy) tuples
    Returns: complex array of shape [num_subcarriers]
    """
    i = grid_to_idx[tx_grid]
    j = grid_to_idx[rx_grid]
    assert i != j, f"tx and rx are the same position: {tx_grid}"
    return H_full[j, i]

def np_get_all_from(H_full: np.ndarray, grid_to_idx: dict, tx_grid: tuple) -> dict:
    """All channels from a given tx position to every other valid position."""
    i = grid_to_idx[tx_grid]
    return {
        idx_to_grid[j]: H_full[j, i]
        for j in range(N) if j != i
    }

def np_get_neighbors_csi(H_full: np.ndarray, grid_to_idx: dict, tx_grid: tuple, radius: int = 1) -> dict:
    """Only get CSI to nearby grid positions (useful for local D2D)."""
    gx, gy = tx_grid
    result = {}
    for rx_grid in valid_2d:
        rx_gx, rx_gy = rx_grid
        if rx_grid == tx_grid:
            continue
        if abs(rx_gx - gx) <= radius and abs(rx_gy - gy) <= radius:
            result[rx_grid] = np_get_csi(H_full, grid_to_idx, tx_grid, rx_grid)
    return result



In [ ]:
#| export

# Assuming df has columns: tx_grid, rx_grid, csi
# and tuples are stored as (gx, gy)

# Build a lookup index from the dataframe once
def build_csi_index(csi_data: pd.DataFrame) -> dict:
    return {
        (tuple(tx), tuple(rx)): csi
        for tx, rx, csi in zip(csi_data["tx_grid"], csi_data["rx_grid"], csi_data["csi"])
    }

def df_get_csi(csi_data: pd.DataFrame, tx_grid: tuple, rx_grid: tuple) -> complex:
    """Get CSI for a single tx->rx pair."""
    # Normalize to plain int tuples (in case of np.int64)
    csi_index = build_csi_index(csi_data)
    tx = (int(tx_grid[0]), int(tx_grid[1]))
    rx = (int(rx_grid[0]), int(rx_grid[1]))
    assert tx != rx, f"tx and rx are the same position: {tx}"
    return csi_index[(tx, rx)]

def df_get_all_from(csi_data: pd.DataFrame, tx_grid: tuple) -> pd.DataFrame:
    """All channels from a given tx position to every other valid position."""
    tx = (int(tx_grid[0]), int(tx_grid[1]))
    return csi_data[csi_data["tx_grid"].apply(lambda t: (int(t[0]), int(t[1])) == tx)]

def df_get_neighbors_csi(csi_data: pd.DataFrame, tx_grid: tuple, radius: int = 1) -> pd.DataFrame:
    """Only get CSI to nearby grid positions."""
    tx = (int(tx_grid[0]), int(tx_grid[1]))
    gx, gy = tx

    def is_neighbor(rx):
        rx = (int(rx[0]), int(rx[1]))
        return rx != tx and abs(rx[0] - gx) <= radius and abs(rx[1] - gy) <= radius

    mask = (
        csi_data["tx_grid"].apply(lambda t: (int(t[0]), int(t[1])) == tx) &
        csi_data["rx_grid"].apply(is_neighbor)
    )
    return csi_data[mask]

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()